# [센서 실습 30] 알코올 센서 – MQ-3

알코올 가스 센서 **MQ-3**으로 날숨 속 알코올 농도(mg/L)를 측정하고 혈중알코올농도(%)로 환산하는 **교육용 음주 측정기**를 만듭니다. 여러 사람이 차례로 측정할 때의 **최고값 추출과 센서 회복 시간**도 다룹니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
MQ-3은 MQ-2·MQ-7과 같은 **산화주석 반도체 센서**이며, 에탄올(알코올)에 특히 민감합니다.
- 알코올 농도가 높을수록 센서 저항 Rs가 작아지고 출력 전압이 높아짐
- 데이터시트 특성 곡선: **날숨 알코올 0.4 mg/L에서 Rs/R0 = 1**, log-log 그래프에서 직선
- 날숨 알코올(BrAC) → 혈중알코올(BAC) 환산: 일반적으로 **혈액 1mL ≈ 날숨 2,100mL**의 알코올 양으로 봄

$$\text{BAC}(\%) \approx \text{BrAC}(\text{mg/L}) \times 0.21$$

| 혈중알코올농도 | 도로교통법 기준 |
|---|---|
| 0.03% 이상 | 면허 정지 |
| 0.08% 이상 | 면허 취소 |

> ⚠️ 저가 반도체 센서는 온도·습도·다른 가스(구강 청결제, 빵 냄새 등)의 영향을 크게 받습니다. **교육용 원리 실습이며, 운전 가능 여부 판단에 절대 사용하면 안 됩니다.**

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 가스 | 에탄올(알코올) |
| 감지 범위 | 0.04 ~ 4 mg/L (날숨) |
| 히터 전원 | 5V, 약 150mA |
| 예열 | 최초 24시간 이상, 매번 수 분 |
| 출력 | 아날로그 (+ 디지털 D0) |

## 2. 라즈베리파이 연결 방법
**준비물**: MQ-3 모듈, MCP3008, 저항 10kΩ·20kΩ, 일회용 빨대(위생), 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| MQ-3 VCC / GND | 5V / GND | 히터 5V |
| MQ-3 A0 → 10kΩ → MCP3008 CH2 | CH2 ↔ GND 사이 20kΩ | 5V 출력을 2/3로 낮춤 |



**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
ADC → 출력 전압 → Rs → Rs/R0 → 날숨 알코올(mg/L) → 혈중알코올(%) 순서로 계산합니다.

In [ ]:
VC, RL, R0 = 5.0, 200.0, 60.0     # 전원(V), 부하저항(kΩ, MQ-3 모듈), 0.4mg/L에서 보정한 R0(kΩ)
DIVIDER = 1.5
M_ALC, B_ALC = -0.66, -0.263      # log10(Rs/R0) = m·log10(mg/L) + b  (0.4mg/L → Rs/R0 = 1)

def adc_to_brac(adc_value, vref=3.3):
    v_out = max(adc_value * vref * DIVIDER, 1e-3)
    rs = (VC - v_out) / v_out * RL
    return 10 ** ((np.log10(rs / R0) - B_ALC) / M_ALC)        # mg/L

def brac_to_bac(mg_l):
    return mg_l * 0.21                                        # %

for a in (0.05, 0.2, 0.35, 0.5):
    m = adc_to_brac(a)
    print(f"ADC {a:.2f} → 날숨 {m:.3f} mg/L → 혈중 {brac_to_bac(m):.3f}%")

### 4-2. 실제 센서 클래스
ADC 2번 채널을 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMQ3:
    def __init__(self, channel=2):
        from gpiozero import MCP3008
        self.adc = MCP3008(channel=channel)

    def read(self):
        v = self.adc.value
        m = adc_to_brac(v)
        return {"adc": round(v, 4), "brac_mgl": round(m, 4), "bac_pct": round(brac_to_bac(m), 4)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 시간(초) | 가정한 상황 | 혈중알코올 (가정) |
|---|---|---|
| 0 ~ 10 | 깨끗한 공기 (기준) | – |
| 10 ~ 15 | A가 5초 불기 | 0.000% (술 안 마심) |
| 30 ~ 35 | B가 5초 불기 | 0.045% |
| 55 ~ 60 | C가 5초 불기 | 0.095% |
| 불고 난 뒤 | 센서가 원래대로 돌아오는 데 시간이 걸림 (회복 시정수 약 5초) | 반도체 센서 특성 |

가상 센서는 날숨 알코올을 **특성 곡선을 거꾸로 적용해 ADC 값**으로 만들고, 반응·회복 속도도 흉내 냅니다.

In [ ]:
class VirtualMQ3:
    BLOWS = [(10, 15, "A", 0.000), (30, 35, "B", 0.045), (55, 60, "C", 0.095)]

    def __init__(self, dt=0.2, seed=30):
        self.dt, self.t, self.level = dt, 0.0, 0.01           # level: 센서가 느끼는 mg/L
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def brac_to_adc(mg_l, vref=3.3):
        rs = 10 ** (M_ALC * np.log10(mg_l) + B_ALC) * R0
        return VC * RL / (RL + rs) / DIVIDER / vref

    def read(self):
        target, who = 0.01, ""                                  # 깨끗한 공기 ≈ 0.01 mg/L
        for a, b, name, bac in self.BLOWS:
            if a <= self.t < b:
                target, who = max(0.01, bac / 0.21), name
        tau = 1.0 if target > self.level else 5.0              # 반응은 빠르고 회복은 느림
        self.level += (target - self.level) * (1 - np.exp(-self.dt / tau))
        adc = round(float(self.brac_to_adc(self.level * np.exp(self.rng.normal(0, 0.03)))), 4)
        self.t += self.dt
        m = adc_to_brac(adc)
        return {"adc": adc, "brac_mgl": round(m, 4), "bac_pct": round(brac_to_bac(m), 4), "who": who}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMQ3(dt=0.2)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
0.2초 간격으로 80초 동안 측정합니다. (가상 센서는 누가 불고 있는지 `who`도 기록합니다.)

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 400               # 측정 횟수
INTERVAL = 0.2               # 측정 간격(초)

sensor = RealMQ3() if USE_REAL_SENSOR else VirtualMQ3(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 사람마다 **불기 시작 ~ 끝난 뒤 3초** 동안의 **최고값**을 측정값으로 씁니다. (날숨 끝부분의 폐 깊은 곳 공기가 가장 정확)
2. 도로교통법 기준으로 판정합니다. (교육용)
3. 다음 사람이 측정하기 전 센서가 기준 수준으로 **회복**되었는지 확인합니다.

In [ ]:
def law(bac):
    return "취소 수준" if bac >= 0.08 else "정지 수준" if bac >= 0.03 else "기준 미만"

base = df[df.time_s < 10].bac_pct.mean()
print(f"깨끗한 공기 기준값: {base:.4f}%\n")
for a, b, name, true_bac in VirtualMQ3.BLOWS:
    seg = df[(df.time_s >= a) & (df.time_s < b + 3)]
    peak = seg.bac_pct.max()
    after = df[(df.time_s >= b + 10) & (df.time_s < b + 11)].bac_pct.mean()
    print(f"{name}: 최고 {peak:.3f}% → {law(peak)}   (10초 뒤 {after:.3f}% {'회복됨' if after < 0.01 else '아직 높음 – 대기 필요'})")
print("\n※ 교육용 원리 실습입니다. 실제 음주 여부 판단에 사용하면 안 됩니다.")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df.time_s, df.bac_pct, "k-", lw=1.5, label="환산 혈중알코올")
for a, b, name, _ in VirtualMQ3.BLOWS:
    ax.axvspan(a, b, color="0.92"); ax.text(a + 0.5, 0.105, name, fontsize=10)
ax.axhline(0.03, color="gray", ls="--", label="0.03% 정지 기준")
ax.axhline(0.08, color="gray", ls=":", label="0.08% 취소 기준")
ax.set_ylim(0, 0.115); ax.set_title("MQ-3 알코올 측정 (교육용)"); ax.set_xlabel("시간 (초)"); ax.set_ylabel("%")
ax.grid(alpha=0.3); ax.legend(loc="upper left"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "알코올센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MQ-3",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/alcohol", payload)
# client.disconnect()

## 직접 해 보기
- 회복 시정수(5초)를 15초로 바꾸면 C가 B 바로 뒤에 측정할 때 어떤 문제가 생기나요?
- R0 보정값이 10% 틀리면 결과가 얼마나 달라지는지 계산해 보세요.
- 측정값이 기준을 넘으면 차량 시동을 막는 '시동 잠금 장치'의 동작 순서를 설계해 보세요.